# 爬虫实验 — Scrapy 采集长春光机所新闻

**目标站点：** [中国科学院长春光学精密机械与物理研究所](https://ciomp.cas.cn/)  
**采集内容：** 综合新闻、科研进展、要闻播报  

**实验流程：**

| 步骤 | 内容 | 掌握技能 |
|------|------|----------|
| 1 | 环境验证 | Python / Scrapy / 依赖管理 |
| 2 | 创建 Scrapy 项目 | 项目结构与组件 |
| 3 | 分析目标网页 | HTML 结构 / CSS 选择器 |
| 4 | 编写 Spider | 数据提取 / 翻页逻辑 |
| 5 | 正则验证 | 数据清洗与校验 |
| 6 | 持久化存储 | Pipeline / 导出 Excel |
| 7 | 运行与查看结果 | 执行爬虫 / Pandas 分析 |

---
## 1. 环境配置与验证

在开始版本检查之前，先完成环境配置（推荐使用 `uv`）。

### 1.1 环境配置（先做）

1. 先安装 `uv`

```powershell
python -m pip install -U uv
```

2. 确认 `uv` 已安装成功

```powershell
uv --version
```

3. 在项目根目录安装/同步依赖（会自动创建 `.venv`）

```powershell
uv sync
```

4. 在 VS Code 的 Notebook 中选择项目虚拟环境

- 点击右上角 `Select Kernel`
- 选择 `data-preprocessing (.venv)`（或路径包含 `.venv\\Scripts\\python.exe`）

### 1.2 环境验证

完成上述配置后，再运行下一段代码检查环境是否就绪。

| 工具/库 | 用途 |
|---------|------|
| **Python 3.11+** | 编程语言 |
| **Scrapy** | 爬虫框架，负责请求调度、页面下载、数据提取 |
| **requests** | HTTP 库，用于交互式页面分析（非 Scrapy 核心依赖） |
| **Pandas** | 数据分析，用于查看和统计爬取结果 |
| **openpyxl** | Excel 读写，Pipeline 用它保存数据 |

In [1]:
import sys
print(f"Python: {sys.version}")

import scrapy
print(f"Scrapy: {scrapy.__version__}")

import pandas as pd
print(f"Pandas: {pd.__version__}")

import openpyxl
print(f"Openpyxl: {openpyxl.__version__}")

print("\n环境就绪 ✓")

Python: 3.11.15 (main, Mar 10 2026, 18:12:25) [MSC v.1944 64 bit (AMD64)]
Scrapy: 2.14.1
Pandas: 3.0.1
Openpyxl: 3.1.5

环境就绪 ✓


---
## 2. Scrapy 项目结构

### 2.1 Scrapy 架构概览

Scrapy 是一个**异步**爬虫框架，核心组件之间的数据流如下：

```
                        ┌───────────┐
                   ②    │ Scheduler │   ③
             ┌─────────>│ (请求队列) │───────────┐
             │          └───────────┘           │
             │                                  ▼
       ┌──────────┐                      ┌────────────┐
   ①   │  Spider  │                      │ Downloader │
START─>│ (爬虫逻辑)│<─────────────────────│ (页面下载)  │
       └──────────┘    ④ Response        └────────────┘
             │
             │ ⑤ Items
             ▼
       ┌──────────┐
       │ Pipeline │
       │(数据管道) │
       └──────────┘
             │
             ▼
         💾 Excel
```

| 步骤 | 说明 |
|------|------|
| ① | Spider 生成初始 Request（起始 URL） |
| ② | Engine 将 Request 交给 Scheduler 排队 |
| ③ | Scheduler 取出 Request，交给 Downloader 下载 |
| ④ | Downloader 下载页面，将 Response 返回给 Spider |
| ⑤ | Spider 解析 Response，产出 **Item**（数据）或新的 **Request**（翻页/详情页） |
| ⑥ | Item 经过 Pipeline 处理后持久化存储 |

### 2.2 项目文件说明

```
ciomp/
├── scrapy.cfg              # Scrapy 部署配置（通常不需要修改）
└── ciomp/                  # Python 包
    ├── __init__.py
    ├── items.py            # ⬅ 数据模型：定义要提取哪些字段
    ├── pipelines.py        # ⬅ 数据管道：拿到数据后怎么处理（存 Excel）
    ├── settings.py         # ⬅ 全局设置：并发数、延迟、启用哪些组件
    ├── middlewares.py       # 中间件（本实验不涉及）
    └── spiders/            # 爬虫目录
        ├── __init__.py
        └── news.py         # ⬅ 爬虫逻辑：从哪抓、怎么解析、怎么翻页
```

> 我们需要编写的文件只有 4 个：`items.py`、`spiders/news.py`、`pipelines.py`、`settings.py`

In [2]:
from pathlib import Path
import sys

project_root = Path("ciomp")

if not (project_root / "scrapy.cfg").exists():
    !{sys.executable} -m scrapy startproject ciomp
else:
    print("项目已存在，跳过创建")

print("\nciomp/ 目录内容：")
for path in sorted(project_root.iterdir()):
    suffix = "/" if path.is_dir() else ""
    print(f"- {path.name}{suffix}")

项目已存在，跳过创建

ciomp/ 目录内容：
- ciomp/
- ciomp_news.xlsx
- scrapy.cfg
- ~$ciomp_news.xlsx


---
## 3. 分析目标网页结构（HTML & CSS）

### 3.1 HTML 基础

网页由 HTML 标签构成，标签可以**嵌套**，形成树形结构：

```html
<标签名 属性="值">内容</标签名>
```

示例：
```html
<a href="news.html" class="db">新闻标题</a>
 │  │               │           │
 标签  属性href       属性class    文本内容
```

常见标签：

| 标签 | 含义 | 示例 |
|------|------|------|
| `<div>` | 块级容器（用来分组） | `<div class="title">标题</div>` |
| `<a>` | 超链接 | `<a href="url">点击</a>` |
| `<ul>` / `<li>` | 无序列表 / 列表项 | `<ul><li>项目1</li></ul>` |
| `<img>` | 图片（自闭合标签） | `<img src="photo.jpg"/>` |
| `<span>` | 行内容器 | `<span class="date">3月</span>` |

### 3.2 CSS 选择器 — 基础语法

CSS 选择器用于**定位**页面中的元素。以下是基本构件：

| 选择器 | 含义 | 示例 | 匹配什么 |
|--------|------|------|---------|
| `tag` | 按标签名 | `div` | 所有 `<div>` |
| `.class` | 按 class 属性 | `.title` | 所有 `class="title"` 的元素 |
| `tag.class` | 标签 + class | `ul.tuwen-item` | `class="tuwen-item"` 的 `<ul>` |
| `#id` | 按 id 属性 | `#header` | `id="header"` 的元素 |
| `*` | 通配符，任意元素 | `*` | 所有元素 |

### 3.3 CSS 选择器 — 层级关系

HTML 标签是嵌套的，选择器可以描述层级关系：

```html
<ul class="tuwen-item">        ← 祖先
  <li>                          ← 子元素（直接）
    <a class="db" href="...">   ← 后代（间接）
      <div class="title">       ← 更深层后代
        标题文字
      </div>
    </a>
  </li>
</ul>
```

| 选择器 | 含义 | 示例 | 说明 |
|--------|------|------|------|
| `A B` | A 的后代 B（任意层级） | `ul.tuwen-item .title` | `<ul>` 内**任意深度**的 `.title` |
| `A > B` | A 的**直接**子元素 B | `ul.tuwen-item > li` | `<ul>` 的**直接** `<li>` 子元素 |

### 3.4 CSS 选择器 — 提取内容

选中元素后，还需要告诉 Scrapy **提取什么**：

| 伪元素 | 提取什么 | 示例 | 结果 |
|--------|---------|------|------|
| `::text` | 元素的文本内容 | `.title::text` | `"新闻标题"` |
| `::attr(x)` | 元素的属性值 | `a::attr(href)` | `"./202603/t20260306_xxx.html"` |
| （不加） | 元素的完整 HTML | `.title` | `<div class="title">新闻标题</div>` |

### 3.5 本实验用到的选择器逐条解析

下面是代码中**实际用到**的每一条选择器，逐个拆解：

```python
# ① 选中列表容器下的所有 <li>
sel.css("ul.tuwen-item > li")
#        ──────────── ─ ──
#        tag.class    > 直接子元素 li

# ② 提取所有新闻标题的文字
sel.css("ul.tuwen-item .title::text")
#        ──────────── ────── ──────
#        列表容器      后代.title  提取文字

# ③ 提取链接地址
li.css("a.db::attr(href)")
#       ──── ────────────
#       <a class="db">  提取 href 属性值

# ④ 提取缩略图地址
li.css("img::attr(src)")
#       ─── ───────────
#       <img> 标签  提取 src 属性值

# ⑤ 提取详情页的所有文字（通配符）
body.css("*::text")
#         ─ ──────
#         任意标签  提取文字
#         （把 <p>、<span>、<div> 等所有标签的文字都提取出来）

# ⑥ 定位翻页区域的链接
response.css("div.pages a.over")
#              ───────── ──────
#              <div class="pages"> 内部的 <a class="over">
```

### 3.6 Scrapy Selector API

| 方法 | 返回类型 | 用途 | 示例 |
|------|----------|------|------|
| `.css("选择器")` | SelectorList | 选中所有匹配元素 | `sel.css("li")` → 所有 `<li>` |
| `.get()` | `str` 或 `None` | 取**第一个**值 | `sel.css(".title::text").get()` → `"标题1"` |
| `.getall()` | `list[str]` | 取**所有**值 | `sel.css(".title::text").getall()` → `["标题1", "标题2", ...]` |
| `.get(default="")` | `str` | 取第一个，没有则返回默认值 | 避免 `None` 导致后续 `.strip()` 报错 |

### 3.7 动手实践

> 在浏览器中打开 https://ciomp.cas.cn/zhxw/ ，按 **F12** 打开开发者工具，切到 **Elements** 面板同步对照。

下面我们先用 `requests` 获取页面，再用 `scrapy.Selector` 练习上述选择器：

In [3]:
import requests
from scrapy import Selector

# 获取综合新闻列表页
resp = requests.get("https://ciomp.cas.cn/zhxw/", timeout=10)
resp.encoding = resp.apparent_encoding

print(f"状态码: {resp.status_code}")
print(f"编码: {resp.encoding}")

状态码: 200
编码: utf-8


In [4]:
sel = Selector(text=resp.text)

# 查看第一条新闻的完整 HTML
first = sel.css("ul.tuwen-item > li")[0]
print(first.get())

<li class="">
                                        <a href="./202603/t20260313_8159082.html" target="_blank" class="db" data-img="">
                                            <div class="img-boxs">
                                                <img src="">
                                            </div>
                                            <div class="tuwen-list">
                                                <div class="title">长春光机所党委理论学习中心组开展树立和践行正确政绩观专题学习</div>
                                                <div class="des overfloat-dot-2"></div>
                                                <div class="date-s">2026-03-13</div>
                                            </div>
                                        </a>
                                    </li>


**页面结构解析：**

```html
<ul class="tuwen-item">           <!-- 新闻列表容器 -->
  <li>
    <a class="db" href="./202603/t20260306_xxx.html">
      <div class="img-boxs">
        <img src="..."/>           <!-- 缩略图 -->
      </div>
      <div class="tuwen-list">
        <div class="title">标题</div>    <!-- 新闻标题 -->
        <div class="des">摘要</div>      <!-- 内容摘要 -->
        <div class="date-s">日期</div>   <!-- 发布日期 -->
      </div>
    </a>
  </li>
</ul>

<div class="pages">               <!-- 分页导航 -->
  <span class="active">1</span>   <!-- 当前页 -->
  <a href="index_1.html">2</a>    <!-- 第2页 -->
  <a href="index_1.html">下一页</a>
</div>
```

### 3.5 练习选择器

下面用上面学到的选择器语法，从真实页面中提取数据。

**思路：** 从外到内逐层定位——先找到列表容器 `ul.tuwen-item`，再深入到每个 `<li>` 中提取 `.title`、`.date-s` 等字段。

In [5]:
# 练习 CSS 选择器：提取所有新闻标题
titles = sel.css("ul.tuwen-item .title::text").getall()
for i, t in enumerate(titles, 1):
    print(f"{i}. {t.strip()}")

1. 长春光机所党委理论学习中心组开展树立和践行正确政绩观专题学习
2. 长春光机所召开树立和践行正确政绩观学习教育启动部署会
3. 长春光机所召开2026年全国两会精神学习传达会
4. 长春光机所离退中心举行三八妇女节手工活动
5. 长春光机所离退中心开展“庆元宵猜灯谜”活动
6. 长春光机所除夕走访慰问在岗职工
7. 长春光机所开展“传承与关爱”春节送温暖慰问活动
8. 长春光机所成功举办“迎春纳福·光启新程”春节系列活动
9. 长春光机所召开2025年统战工作交流会暨新年团拜会
10. 长春光机所离退中心开展“迎新春写春联拓福印”活动
11. 长春光机所2026年度职业健康安全管理体系内审员培训班成功举办
12. 长春光机所召开研究生思政工作领导小组工作会议


### 3.6 列表页 → 详情页

新闻网站通常分两层结构：
- **列表页**：只有标题 + 日期 + 链接（信息不完整）
- **详情页**：包含完整正文

所以爬虫需要**两步走**：先从列表页提取链接，再跟进到详情页提取正文。

```
列表页                              详情页
┌─────────────────┐    follow    ┌─────────────────┐
│ 标题1  日期  链接 |─────────────>│ 标题            │
│ 标题2  日期  链接 │              │ 正文正文正文...  │
│ 标题3  日期  链接 │              │                 │
│ ...             │              └─────────────────┘
│ [下一页]         │
└─────────────────┘
```

下面我们手动模拟这个过程——取第一条新闻的链接，访问详情页，提取正文：

In [6]:
# 练习提取多个字段
for li in sel.css("ul.tuwen-item > li")[:3]:
    title = li.css(".title::text").get(default="").strip()
    date  = li.css(".date-s::text").get(default="").strip()
    href  = li.css("a.db::attr(href)").get(default="")
    print(f"[{date}] {title}")
    print(f"  链接: {href}\n")

[2026-03-13] 长春光机所党委理论学习中心组开展树立和践行正确政绩观专题学习
  链接: ./202603/t20260313_8159082.html

[2026-03-13] 长春光机所召开树立和践行正确政绩观学习教育启动部署会
  链接: ./202603/t20260313_8159077.html

[2026-03-13] 长春光机所召开2026年全国两会精神学习传达会
  链接: ./202603/t20260313_8159075.html



In [7]:
# 分析详情页结构
detail_url = sel.css("ul.tuwen-item a.db::attr(href)").get()
detail_url = requests.compat.urljoin("https://ciomp.cas.cn/zhxw/", detail_url)

resp2 = requests.get(detail_url, timeout=10)
resp2.encoding = resp2.apparent_encoding
sel2 = Selector(text=resp2.text)

# 正文在 .trs_editor_view 中
content = sel2.css(".trs_editor_view *::text").getall()
content_text = "\n".join(t.strip() for t in content if t.strip())
print(f"正文长度: {len(content_text)} 字\n")
print(content_text[:300])

正文长度: 663 字

2026年3月13日，长春光机所党委理论学习中心组召开专题学习会,深入学习贯彻习近平总书记关于树立和践行正确政绩观的重要论述，扎实推进树立和践行正确政绩观学习教育走深走实，切实把思想和行动统一到党中央决策部署上来，以正确政绩观引领研究所高质量发展，为实现高水平科技自立自强筑牢思想根基。党委书记、副所长金宏主持会议。
副所长韩诚山领学习近平总书记关于树立和践行正确政绩观的重要论述，中心组成员围绕习近平总书记的重要论述开展集中学习研讨，深入领会政绩观的核心内涵，深刻把握“政绩为谁而树、树什么样的政绩、靠什么树政绩”的关键问题。大家一致认为，政绩观是世界观、人生观、价值观在从政行为中的具体体现，正确


---
## 4. 编写 Spider

### 4.1 定义数据模型 (Items)

`Item` 是 Scrapy 的数据容器，类似数据库的表结构——定义好字段后，Spider 提取的数据和 Pipeline 处理的数据就有了统一的"契约"：

```
Spider 产出 Item ──→ Pipeline 接收 Item ──→ 写入 Excel
     ↑                        ↑
     └──── 字段名必须一致 ──────┘
```

In [8]:
%%writefile ciomp/ciomp/items.py
import scrapy


class NewsItem(scrapy.Item):
    """新闻数据模型。"""
    section = scrapy.Field()     # 所属栏目
    title = scrapy.Field()       # 标题
    date = scrapy.Field()        # 发布日期
    url = scrapy.Field()         # 文章链接
    image = scrapy.Field()       # 缩略图 URL
    content = scrapy.Field()     # 正文

Overwriting ciomp/ciomp/items.py


### 4.2 编写新闻 Spider

Spider 是 Scrapy 的核心，负责：**发起请求 → 解析响应 → 产出数据或新请求**。

我们的 Spider 包含 3 个方法，对应 3 个阶段：

```
start()               parse_list()                 parse_detail()
  │                        │                             │
  │ 生成 3 个栏目 URL       │ 解析列表页                    │ 解析详情页
  │                        │  ├─ 提取每条新闻的链接         │  ├─ 提取正文
  ▼                        │  ├─ yield Request ─────────>│   ├─ 正则验证日期
3 个 Request               │  └─ 找"下一页" ──┐            │  └─ yield Item ──→ Pipeline
                           │                 │           │
                           │<────────────────┘           │
                           │   (递归翻页)                  │
```

**关键概念：**

| 概念 | 说明 |
|------|------|
| `yield Request` | 告诉 Scrapy"请下载这个 URL，完成后调用指定的回调函数" |
| `yield Item` | 告诉 Scrapy"这是一条提取好的数据，交给 Pipeline 处理" |
| `response.follow(href)` | 自动将相对路径（如 `./202603/...`）拼接为完整 URL |
| `cb_kwargs` | 向回调函数传递额外参数（如栏目名、标题），避免重复提取 |

In [9]:
%%writefile ciomp/ciomp/spiders/news.py
import re
import scrapy
from ciomp.items import NewsItem


class NewsSpider(scrapy.Spider):
    name = "news"
    allowed_domains = ["ciomp.cas.cn"]

    # 栏目配置
    sections = {
        "综合新闻": "https://ciomp.cas.cn/zhxw/",
        "科研进展": "https://ciomp.cas.cn/kydt/",
        "要闻播报": "https://ciomp.cas.cn/ywbb/",
    }

    async def start(self):
        for section_name, url in self.sections.items():
            yield scrapy.Request(
                url,
                callback=self.parse_list,
                cb_kwargs={"section": section_name},
            )

    def parse_list(self, response, section):
        """解析新闻列表页。"""
        for li in response.css("ul.tuwen-item > li"):
            href = li.css("a.db::attr(href)").get()
            if not href:
                continue

            # 跳过非 HTML 资源（.doc / .pdf 等）
            if re.search(r"\.(doc|docx|pdf|xls|xlsx|zip)$", href, re.I):
                continue

            meta_data = {
                "section": section,
                "title": li.css(".title::text").get(default="").strip(),
                "date": li.css(".date-s::text").get(default="").strip(),
                "image": li.css("img::attr(src)").get(default=""),
            }

            yield response.follow(
                href,
                callback=self.parse_detail,
                cb_kwargs=meta_data,
            )

        # 自动翻页：找"下一页"链接
        next_page = response.css("div.pages a.over")
        for a in next_page:
            if a.css("::text").get("").strip() == "下一页":
                yield response.follow(
                    a,
                    callback=self.parse_list,
                    cb_kwargs={"section": section},
                )
                break

    def parse_detail(self, response, section, title, date, image):
        """解析文章详情页，提取正文。"""
        # 正文区域
        body_sel = response.css(".trs_editor_view")
        if not body_sel:
            body_sel = response.css(".wrap")

        paragraphs = body_sel.css("*::text").getall()
        content = "\n".join(p.strip() for p in paragraphs if p.strip())

        # ---------- 正则验证 ----------
        # 验证日期格式：YYYY-MM-DD
        if date and not re.match(r"^\d{4}-\d{2}-\d{2}$", date):
            self.logger.warning(f"日期格式异常: {date!r} ({title})")
            date = ""  # 清空不合规的日期

        # 清理标题中的多余空白
        title = re.sub(r"\s+", " ", title).strip()

        item = NewsItem(
            section=section,
            title=title,
            date=date,
            url=response.url,
            image=image,
            content=content,
        )
        yield item

Overwriting ciomp/ciomp/spiders/news.py


### 4.3 正则表达式（RegEx）

正则表达式用于**模式匹配**，在爬虫中常用来验证和清洗数据。

### 4.3.1 基础语法

| 符号 | 含义 | 示例 | 匹配 |
|------|------|------|------|
| `\d` | 一个数字 (0-9) | `\d` | `3`、`0` |
| `\d{n}` | 恰好 n 个数字 | `\d{4}` | `2026`（4位） |
| `\s` | 空白字符（空格、换行、Tab） | `\s` | ` `、`\n` |
| `.` | 任意一个字符（除换行） | `a.b` | `a1b`、`a-b`、`axb` |
| `\.` | **转义**：匹配真正的点号 `.` | `\.doc` | `.doc`（而非 `xdoc`） |
| `+` | 前面的元素出现 **1次或多次** | `\d+` | `1`、`123`、`99999` |
| `*` | 前面的元素出现 **0次或多次** | `\d*` | `` (空)、`123` |
| `^` | 字符串**开头** | `^\d` | 以数字开头 |
| `$` | 字符串**结尾** | `\.html$` | 以 `.html` 结尾 |
| `(A\|B)` | A 或 B | `(doc\|pdf)` | `doc` 或 `pdf` |

> **关键区别：** `.` 匹配任意字符，`\.` 匹配字面点号。这是初学者最常犯的错误。

### 4.3.2 Python `re` 模块

Python 通过 `re` 模块使用正则表达式，提供了 3 个常用函数：

| 函数 | 作用 | 返回值 | 适用场景 |
|------|------|--------|---------|
| `re.match(pattern, s)` | 从**开头**匹配 | Match 对象 或 None | 验证格式是否合规 |
| `re.search(pattern, s)` | 在**任意位置**搜索 | Match 对象 或 None | 检查是否包含某模式 |
| `re.sub(pattern, repl, s)` | 查找并**替换** | 替换后的字符串 | 清洗数据 |
| `re.compile(pattern)` | **预编译**正则 | Pattern 对象 | 同一正则用多次时提高性能 |

**`re.match` vs `re.search` 区别：**
```python
text = "文件: report.doc"

re.match(r"report", text)   # → None      （开头是"文件"，不是"report"）
re.search(r"report", text)  # → Match!    （在字符串中间找到了）
```

**标志位：**
| 标志 | 含义 | 示例 |
|------|------|------|
| `re.I` | 忽略大小写 | `re.search(r"\.doc$", "FILE.DOC", re.I)` → 匹配 |

### 4.3.3 本实验用到的正则逐条解析

Spider 代码中用了 3 条正则，逐条拆解：

```python
# ① 验证日期格式：必须是 YYYY-MM-DD
re.match(r"^\d{4}-\d{2}-\d{2}$", date)
#           │ ────  ─  ────  ─  ──── │
#           │ 4位数 - 2位数 -  2位数  │
#           ^开头                    $结尾
#
# ✓ "2026-03-06"    ✗ "2026-3-6"    ✗ "3月6日"

# ② 清理标题中的多余空白
re.sub(r"\s+", " ", title)
#         ───  ───  ─────
#         匹配  替换  原字符串
#         连续   为
#         空白  单空格
#
# "标题  换行\n测试" → "标题 换行 测试"

# ③ 过滤非 HTML 链接（列表页中有些链接指向 .doc 文件）
re.search(r"\.(doc|docx|pdf|xls|xlsx|zip)$", href, re.I)
#            ── ─────────────────────────  ─       ────
#            \.  6种文件扩展名(任选其一)    $结尾   忽略大小写
#          转义点号                        
#
# ✓ "P020250829.doc"    ✗ "t20260306.html"
```

可以在下方交互测试所有 3 条正则：

In [10]:
import re

# ---- ① 验证日期格式 ----
print("=== 日期验证: re.match(r'^\\d{4}-\\d{2}-\\d{2}$', date) ===")
date_pattern = re.compile(r"^\d{4}-\d{2}-\d{2}$")
for d in ["2026-03-06", "2026-3-6", "3月6日", "20260306", ""]:
    result = "✓" if date_pattern.match(d) else "✗"
    print(f"  {result}  {d!r}")

# ---- ② 清理空白 ----
print("\n=== 清理空白: re.sub(r'\\s+', ' ', text) ===")
for text in ["标题  多余空格", "标题\n换行\t制表符", "正常标题"]:
    cleaned = re.sub(r"\s+", " ", text).strip()
    print(f"  {text!r:30s} → {cleaned!r}")

# ---- ③ 过滤非 HTML 链接 ----
print("\n=== 文件过滤: re.search(r'\\.(doc|pdf|xls|...)$', href, re.I) ===")
for href in ["t20260306_xxx.html", "P020250829.doc", "report.PDF", "data.xlsx", "index_1.html"]:
    hit = re.search(r"\.(doc|docx|pdf|xls|xlsx|zip)$", href, re.I)
    result = "✗ 跳过" if hit else "✓ 保留"
    print(f"  {result}  {href}")

=== 日期验证: re.match(r'^\d{4}-\d{2}-\d{2}$', date) ===
  ✓  '2026-03-06'
  ✗  '2026-3-6'
  ✗  '3月6日'
  ✗  '20260306'
  ✗  ''

=== 清理空白: re.sub(r'\s+', ' ', text) ===
  '标题  多余空格'                     → '标题 多余空格'
  '标题\n换行\t制表符'                  → '标题 换行 制表符'
  '正常标题'                         → '正常标题'

=== 文件过滤: re.search(r'\.(doc|pdf|xls|...)$', href, re.I) ===
  ✓ 保留  t20260306_xxx.html
  ✗ 跳过  P020250829.doc
  ✗ 跳过  report.PDF
  ✗ 跳过  data.xlsx
  ✓ 保留  index_1.html


---
## 5. 配置 Settings

`settings.py` 控制 Scrapy 的全局行为。下面逐项说明每个关键配置：

| 配置项 | 值 | 说明 |
|--------|-----|------|
| `ROBOTSTXT_OBEY` | `True` | 遵守网站的 robots.txt 规则（礼貌爬取） |
| `DOWNLOAD_DELAY` | `0.25` | 每次请求间隔 0.25 秒，避免给服务器造成压力 |
| `CONCURRENT_REQUESTS` | `8` | 最多同时发送 8 个请求 |
| `CLOSESPIDER_ITEMCOUNT` | `50` | 抓到 50 条数据后自动停止（课堂演示用） |
| `USER_AGENT` | Chrome UA | 模拟浏览器请求，部分网站会拒绝默认的 Scrapy UA |
| `ITEM_PIPELINES` | `{ExcelPipeline: 300}` | 启用 Excel 管道，300 是优先级（数字越小越先执行） |

> **注意：** `CLOSESPIDER_ITEMCOUNT = 50` 是为了课堂演示快速完成。实际使用时可去掉此限制，爬虫会抓完所有页面后自然停止。

In [11]:
%%writefile ciomp/ciomp/settings.py
BOT_NAME = "ciomp"
SPIDER_MODULES = ["ciomp.spiders"]
NEWSPIDER_MODULE = "ciomp.spiders"

# 礼貌爬取（快速演示配置）
ROBOTSTXT_OBEY = True
DOWNLOAD_DELAY = 0.25
CONCURRENT_REQUESTS = 8

# 课堂演示：抓到 50 条自动停止
CLOSESPIDER_ITEMCOUNT = 50

# 请求头
DEFAULT_REQUEST_HEADERS = {
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "zh-CN,zh;q=0.9",
}
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

# 启用 Pipeline
ITEM_PIPELINES = {
    "ciomp.pipelines.ExcelPipeline": 300,
}

# 输出编码
FEED_EXPORT_ENCODING = "utf-8"

# 日志级别
LOG_LEVEL = "INFO"

REQUEST_FINGERPRINTER_IMPLEMENTATION = "2.7"
TWISTED_REACTOR = "twisted.internet.asyncioreactor.AsyncioSelectorReactor"

Overwriting ciomp/ciomp/settings.py


---
## 6. 持久化存储 — Excel Pipeline

Pipeline 是 Scrapy 的数据处理管道。每当 Spider 产出一个 Item，Pipeline 都会接收并处理它。

**Pipeline 生命周期：**

```
Spider 启动
    │
    ▼
open_spider()          ← 初始化：创建 Excel 工作簿、写入表头
    │
    ▼
┌─────────────────┐
│ process_item()  │ ← 每产出一条 Item 调用一次：将数据写入一行
│     ↕ 循环       │
└─────────────────┘
    │
    ▼
close_spider()         ← 收尾：调整列宽、保存 .xlsx 文件
    │
    ▼
Spider 关闭
```

**要点：**
- `process_item()` **必须** `return item`，否则后续 Pipeline 收不到数据
- 正文截断到 500 字符，避免 Excel 单元格过大
- 文件保存在 Scrapy 的工作目录（`ciomp/`）下

In [12]:
%%writefile ciomp/ciomp/pipelines.py
import logging

from openpyxl import Workbook

logger = logging.getLogger(__name__)


class ExcelPipeline:
    """将爬取结果保存为 Excel 文件。"""

    HEADERS = ["栏目", "日期", "标题", "链接", "正文"]

    def open_spider(self):
        self.wb = Workbook()
        self.ws = self.wb.active
        self.ws.title = "新闻数据"
        self.ws.append(self.HEADERS)
        self.count = 0

    def process_item(self, item):
        self.ws.append([
            item.get("section", ""),
            item.get("date", ""),
            item.get("title", ""),
            item.get("url", ""),
            item.get("content", "")[:500],  # 正文截断，避免单元格过大
        ])
        self.count += 1
        return item

    def close_spider(self):
        # 调整列宽
        col_widths = [10, 12, 40, 50, 60]
        for i, w in enumerate(col_widths, 1):
            self.ws.column_dimensions[
                chr(64 + i)  # A, B, C, D, E
            ].width = w

        filename = "ciomp_news.xlsx"
        self.wb.save(filename)
        logger.info(f"已保存 {self.count} 条数据到 {filename}")

Overwriting ciomp/ciomp/pipelines.py


---
## 7. 运行爬虫

**常用命令：**

| 命令 | 作用 |
|------|------|
| `scrapy list` | 查看项目中已注册的 Spider |
| `scrapy crawl news` | 运行名为 `news` 的 Spider |
| `scrapy crawl news -o data.json` | 运行并额外导出为 JSON |

**日志输出怎么看：**

```
Crawled 96 pages (at 47 pages/min), scraped 51 items (at 28 items/min)
         │                                    │
         页面请求总数                           实际产出的数据条目数
         (列表页 + 详情页 + robots.txt)         (每个详情页产出 1 条 Item)
```

**常见结束原因（`finish_reason`）：**

| 值 | 含义 |
|----|------|
| `closespider_itemcount` | 达到 `CLOSESPIDER_ITEMCOUNT` 设定的上限，自动停止 |
| `finished` | 所有页面爬取完毕，自然结束 |
| `shutdown` | 手动中断（Ctrl+C） |

In [13]:
# 查看已注册的 Spider
!cd ciomp && {sys.executable} -m scrapy list

news


In [14]:
# 运行爬虫
!cd ciomp && {sys.executable} -m scrapy crawl news

2026-03-16 15:15:20 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: ciomp)
2026-03-16 15:15:20 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.11.9',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.0',
 'Twisted': '25.5.0',
 'Python': '3.11.15 (main, Mar 10 2026, 18:12:25) [MSC v.1944 64 bit (AMD64)]',
 'pyOpenSSL': '25.3.0 (OpenSSL 3.5.5 27 Jan 2026)',
 'cryptography': '46.0.5',
 'Platform': 'Windows-10-10.0.26200-SP0'}
2026-03-16 15:15:20 [scrapy.addons] INFO: Enabled addons:
[]
2026-03-16 15:15:20 [scrapy.extensions.telnet] INFO: Telnet Password: c77b5fd669c52557
2026-03-16 15:15:20 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.closespider.CloseSpider',
 'scrapy.extensions.logstats.LogStats']
2026-03-16 15:15:20 [scrapy.crawler] INFO: Overridden settings:
{'BOT_NAME': 'ciomp',
 'CLOSESPIDER_ITEMCOUNT': 50

---
## 8. 查看结果

爬虫产出的数据已保存为 `ciomp/ciomp_news.xlsx`。下面用 Pandas 加载并做基本的统计分析，验证数据质量。

In [15]:
import pandas as pd

df = pd.read_excel("ciomp/ciomp_news.xlsx")
print(f"共采集 {len(df)} 条新闻\n")
df.head(10)

共采集 57 条新闻



,栏目,日期,标题,链接,正文
0,科研进展,2025-04-21,基于对称Littrow结构的双光栅共模误差抑制技术,https://ciomp.cas.cn/zhxw/202504/t20250421_793...,@page{size:8.27in11.69in;margin-left:1.25in;ma...
1,科研进展,2025-05-12,可用于先进人工视觉系统的AlScN/p-i-n GaN异质结紫外光电突触,https://ciomp.cas.cn/kydt/202505/t20250512_793...,@page{size:8.27in11.69in;margin-left:1.25in;ma...
2,科研进展,2025-06-17,Advanced Materials:长春光机所在纤锌矿铁电体研究领域取得重大突破,https://ciomp.cas.cn/zhxw/202506/t20250617_793...,@page{size:8.27in11.69in;margin-left:1.25in;ma...
3,科研进展,2025-07-10,缺陷功能化新突破：调控氮空位加速电子冷却，破解紫外LED注入效率瓶颈,https://ciomp.cas.cn/zhxw/202507/t20250710_793...,"<span data-index=""10"" style=""font-family: 宋体, ..."
4,科研进展,2025-10-15,基于卷积效应修正的离子束加工去除函数在线测量方法,https://ciomp.cas.cn/kydt/202510/t20251015_799...,近期，中国科学院长春光机所在\nLight： Advanced Manufacturing\...
5,科研进展,2025-10-30,长春光机所在傅里叶光子计数时间门控拉曼技术研究取得新进展,https://ciomp.cas.cn/kydt/202510/t20251030_799...,近期，中国科学院长春光机所在\nLight: Science & Applications\...
6,科研进展,2025-11-11,再立新功！长春光机所研制的“天问一号”高分辨率相机成功观测到星际天体阿特拉斯,https://ciomp.cas.cn/kydt/202511/t20251111_800...,"<span data-index=""10"" style=""font-family: 宋体, ..."
7,科研进展,2024-09-18,国家自然科学基金国家重大科研仪器研制项目（部门推荐）“1.5米扫描干涉场曝光系统”后评估现场...,https://ciomp.cas.cn/zhxw/202409/t20240918_793...,2024年9月12日，国家自然科学基金国家重大科研仪器研制项目（部门推荐）“1.5米扫描干涉...
8,科研进展,2024-09-18,国家自然科学基金重大项目“3-5微米中红外波段大功率全光纤化激光器基础研究”中期检查会顺利召开,https://ciomp.cas.cn/zhxw/202409/t20240918_793...,2024年9月12日，国家自然科学基金重大项目“3－5微米中红外波段大功率全光纤化激光器基础...
9,科研进展,2024-11-06,中国科学院长春光机所在飞秒激光制备无涂层持久超疏水表面研究取得进展,https://ciomp.cas.cn/zhxw/202411/t20241106_793...,@page{size:8.27in11.69in;margin-left:1.25in;ma...


In [16]:
# 各栏目文章数量
print("=== 各栏目统计 ===")
print(df["栏目"].value_counts().to_string())

=== 各栏目统计 ===
栏目
科研进展    57


In [17]:
# 按月统计发文趋势
df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
monthly = df.set_index("日期").resample("ME")["标题"].count()
monthly.tail(12)

日期
2024-12-31    0
2025-01-31    3
2025-02-28    0
2025-03-31    0
2025-04-30    1
2025-05-31    1
2025-06-30    1
2025-07-31    1
2025-08-31    0
2025-09-30    0
2025-10-31    2
2025-11-30    1
Freq: ME, Name: 标题, dtype: int64

In [18]:
# 正文长度分布
df["正文长度"] = df["正文"].str.len()
df["正文长度"].describe()

count     57.0
mean     500.0
std        0.0
min      500.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: 正文长度, dtype: float64